In [ ]:
# CELL 1: Mount Drive + Install Dependencies
from google.colab import drive
drive.mount('/content/drive')

!pip install ultralytics roboflow -q
print('Dependencies installed')

In [ ]:
# CELL 2: Download license plate detection dataset from Roboflow
# Go to https://universe.roboflow.com, search "license plate detection",
# pick a dataset, then go to its "Download Dataset" -> YOLOv8 tab.
# It will show you a code snippet with your real API key, workspace, and project name.
# Replace the values below with that exact snippet.

from roboflow import Roboflow

rf = Roboflow(api_key="Y40hwiHPhzZmhSVJL5pk")
project = rf.workspace("kenny-dgv0n").project("license-plate-recognition-rxg4e-lqjmc")
dataset = project.version(1).download("yolov8", location="/content/data/plate_detection")

print("Plate dataset downloaded to:", dataset.location)

In [ ]:
# CELL 3: Verify dataset structure and class names
import os
import yaml

base = dataset.location

for root, dirs, files in os.walk(base):
    print(root, "-", len(files), "files")

yaml_path = os.path.join(base, "data.yaml")
if os.path.exists(yaml_path):
    with open(yaml_path) as f:
        print(f.read())

In [ ]:
# CELL 4: Train YOLOv8 plate detector
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    name="yolo_plate_v1",
    device=0,
    patience=15,
)

In [ ]:
# CELL 5: Validate the trained model (mAP, precision, recall)
metrics = model.val()
print(metrics)

In [ ]:
# CELL 6: Quick visual test on a few validation images
import glob

val_images = glob.glob(f"{dataset.location}/valid/images/*")[:5]
results = model.predict(val_images, save=True, conf=0.4)
print("Predictions saved to runs/detect/predict")

In [ ]:
# CELL 7: Save trained weights + results back to Google Drive
import shutil
import os
import glob

runs_path = "/content/runs/detect/yolo_plate_v1*"
matching_folders = glob.glob(runs_path)

if matching_folders:
    latest_source = max(matching_folders, key=os.path.getmtime)
    folder_name = os.path.basename(latest_source)
    dest = f"/content/drive/MyDrive/alpr-vehicle-system/runs/{folder_name}"

    if os.path.exists(dest):
        shutil.rmtree(dest)
    shutil.copytree(latest_source, dest)

    print("Saved to Drive:")
    print("  ", dest)
    print("Best weights at:", f"{dest}/weights/best.pt")
    print("Copy to your project: models/yolo_plate/best.pt")
else:
    print("No training folder found. Run Cell 4 first.")